In [4]:
import os
import numpy as np
from nilearn import image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [5]:
left_mask_path = "TPJ_refined_mask_L.nii.gz"
right_mask_path = "TPJ_refined_mask_R.nii.gz"

data_dir = "/Users/silvycollin/Documents/research/narrativeProject/21styear_story/data/"
confounds_dir = "/Users/silvycollin/Documents/research/narrativeProject/21styear_story/confounds/"


In [ ]:
# ----------------------------
# Load masks
# ----------------------------
left_mask_orig = image.load_img(left_mask_path)
right_mask_orig = image.load_img(right_mask_path)

# ----------------------------
# Collect 4D participant images
# ----------------------------
img_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".nii.gz")])
n_participants = len(img_files)
print(f"Found {n_participants} participant 4D images")

# ----------------------------
# Reference image (for resampling)
# ----------------------------
ref_img = image.load_img(os.path.join(data_dir, img_files[0]))

# Resample masks to match the first image’s space
left_mask_res = image.resample_to_img(left_mask_orig, ref_img.slicer[..., 0], interpolation='nearest')
right_mask_res = image.resample_to_img(right_mask_orig, ref_img.slicer[..., 0], interpolation='nearest')

left_mask_data = left_mask_res.get_fdata().astype(bool)
right_mask_data = right_mask_res.get_fdata().astype(bool)

n_left_voxels = np.sum(left_mask_data)
n_right_voxels = np.sum(right_mask_data)
n_timepoints = ref_img.shape[-1]
print(f"Each participant has {n_timepoints} timepoints")
print(f"Left TPJ voxels: {n_left_voxels}, Right TPJ voxels: {n_right_voxels}")

# ----------------------------
# Preallocate arrays (voxels × time × participants)
# ----------------------------
left_3d = np.zeros((n_left_voxels, n_timepoints, n_participants))
right_3d = np.zeros((n_right_voxels, n_timepoints, n_participants))

# ----------------------------
# Loop through participants
# ----------------------------
for i, fname in enumerate(img_files):
    img_path = os.path.join(data_dir, fname)
    subj_id = fname.split("_")[0]  # e.g., 'sub-075'
    confound_path = os.path.join(confounds_dir, f"{subj_id}_task-21styear_desc-confounds_regressors.tsv")

    print(f"\nProcessing {subj_id} ({i+1}/{n_participants})")

    # Load image and confounds
    img = image.load_img(img_path)
    confounds_df = pd.read_csv(confound_path, sep="\t")

    # Select confounds
    confound_cols = [c for c in confounds_df.columns if any(x in c for x in [
        "trans_x", "trans_y", "trans_z",
        "rot_x", "rot_y", "rot_z"
    ])]
    confounds = confounds_df[confound_cols].fillna(0).values

    # Clean the image (regress confounds, detrend, standardize)
    clean_data = image.clean_img(
        img,
        confounds=confounds,
        detrend=True,
        standardize=True
    )

    clean_img_data = clean_data.get_fdata()

    # Resample masks to participant space (if needed)
    left_mask_res = image.resample_to_img(left_mask_orig, clean_data.slicer[..., 0], interpolation='nearest')
    right_mask_res = image.resample_to_img(right_mask_orig, clean_data.slicer[..., 0], interpolation='nearest')

    left_mask_data = left_mask_res.get_fdata().astype(bool)
    right_mask_data = right_mask_res.get_fdata().astype(bool)

    # Extract voxel time series (voxels × time)
    left_ts = clean_img_data[left_mask_data, :]
    right_ts = clean_img_data[right_mask_data, :]

    # Store in 3D array
    left_3d[:, :, i] = left_ts
    right_3d[:, :, i] = right_ts

# ----------------------------
# Save outputs
# ----------------------------
np.save("left_TPJ_3D_clean.npy", left_3d)
np.save("right_TPJ_3D_clean.npy", right_3d)

print("Left TPJ array shape:", left_3d.shape, "(voxels × time × subjects)")
print("Right TPJ array shape:", right_3d.shape, "(voxels × time × subjects)")


Found 25 participant 4D images


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_6771/2967690830.py:20: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  left_mask_res = image.resample_to_img(left_mask_orig, ref_img.slicer[..., 0], interpolation='nearest')
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/image/resampling.py:805: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  return resample_img(
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_6771/2967690830.py:21: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  right_mask_res = image.resample_to_img(right_mask_orig, ref_img.slicer[..., 0], interp

Each participant has 2249 timepoints
Left TPJ voxels: 239, Right TPJ voxels: 144

Processing sub-075 (1/25)
